# Low-ℓ BB — Manual Tile Inspector

Inspect raw unmasked patches first, choose a mask, then compare masked patches.

**To switch Stokes component**: change `STOKES_KEY` (T / Q / U) in Paths, re-run from Load map down.

In [ ]:
import os
import sys
import numpy as np
import healpy as hp
import matplotlib
import matplotlib.pyplot as plt
from spt3g import core, maps
from astropy.io import fits
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, _here)

from Plot import (
    apply_spt_style,
    mask_map,
    show_map_full_field,
    show_map_thumbnail,
)

apply_spt_style()

#### Paths

In [ ]:
# Stokes selector
# Change to "Q" or "U" and re-run from §2 to inspect a different component.
STOKES_KEY = "T"

# Map STOKES_KEY to FITS field index (0=T, 1=Q, 2=U)
_STOKES_FIELD = {"T": 0, "Q": 1, "U": 2}
assert STOKES_KEY in _STOKES_FIELD, f"STOKES_KEY must be T / Q / U, got '{STOKES_KEY}'"

# Data paths
DATA_DIR   = "/sptgrid/analysis/spt3g_d1_midell_tqu_healpix"
COADD_PATH = os.path.join(DATA_DIR, "real_data_maps", "full")

full_220ghz = os.path.join(COADD_PATH, "full_220ghz.fits")
full_150ghz = os.path.join(COADD_PATH, "full_150ghz.fits")
full_095ghz = os.path.join(COADD_PATH, "full_095ghz.fits")

# Active frequency — change to full_150ghz or full_095ghz to switch band
COADD_FILE = full_220ghz

# Mask directory
MASK_DIR = "/sptlocal/user/creichardt/bb2020"

# Available masks — refer to this when choosing in §6
MASK_FILES = {
    "mask_250_30"   : "puremask8192_0p5medwt_250mJy_30arcmin.npz",
    "mask_250_60"   : "puremask8192_0p5medwt_250mJy_60arcmin.npz",
    "mask_250_nd30" : "puremask8192_0p5medwt_250mJy_nodisk_30arcmin.npz",
    "mask_250_nd60" : "puremask8192_0p5medwt_250mJy_nodisk_60arcmin.npz",
    "mask_100_30"   : "puremask8192_0p5medwt_100mJy_30arcmin.npz",
    "mask_apod_30"  : "puremask8192_0p5medwt_30arcmin.npz",
    "mask_apod_60"  : "puremask8192_0p5medwt_60arcmin.npz",
}

# Output directory
SAVE_DIR = "/sptlocal/user/vwelke/lowl_bb_tiles"
os.makedirs(SAVE_DIR, exist_ok=True)

# Display parameters
RESO_ARCMIN = 0.5
PATCH_DEG   = 15.0
PATCH_PIX   = int(PATCH_DEG * 60 / RESO_ARCMIN)  # pixels per side
CMAP        = "coolwarm"

print(f"Stokes key  : {STOKES_KEY}  (FITS field index {_STOKES_FIELD[STOKES_KEY]})")
print(f"Active file : {os.path.basename(COADD_FILE)}")
print(f"Patch size  : {PATCH_DEG}° × {PATCH_DEG}°  =  {PATCH_PIX} × {PATCH_PIX} px")
print(f"Resolution  : {RESO_ARCMIN} arcmin/px")

#### Load map

In [ ]:
# Load only the selected Stokes component 
field_idx  = _STOKES_FIELD[STOKES_KEY]
stokes_arr = hp.read_map(COADD_FILE, field=field_idx, partial=False)

nside = hp.get_nside(stokes_arr)
print(f"Loaded     : {STOKES_KEY}  from  {os.path.basename(COADD_FILE)}")
print(f"nside      : {nside}")
print(f"Total pix  : {len(stokes_arr):,}")

#### Observed-pixel mask

In [ ]:
obs_mask = np.isfinite(stokes_arr) & (stokes_arr != hp.UNSEEN)

obs_pix        = np.where(obs_mask)[0]
theta_c, phi_c = hp.pix2ang(nside, obs_pix)
ra_obs  = np.degrees(phi_c)
ra_obs  = np.where(ra_obs > 180, ra_obs - 360, ra_obs)   # shift to [-180, 180]
dec_obs = 90.0 - np.degrees(theta_c)

ra_min,  ra_max  = ra_obs.min(),  ra_obs.max()
dec_min, dec_max = dec_obs.min(), dec_obs.max()

print(f"Observed pixels  : {obs_mask.sum():,}  ({obs_mask.sum()/len(stokes_arr)*100:.2f}% of sky)")
print(f"RA  range  :  {ra_min:.2f}° → {ra_max:.2f}°   span = {ra_max-ra_min:.2f}°")
print(f"Dec range  : {dec_min:.2f}° → {dec_max:.2f}°   span = {dec_max-dec_min:.2f}°")

#### Tile grid

In [ ]:
ra_centres  = np.arange(ra_min  + PATCH_DEG/2, ra_max  + PATCH_DEG/2, PATCH_DEG)
dec_centres = np.arange(dec_min + PATCH_DEG/2, dec_max + PATCH_DEG/2, PATCH_DEG)

n_ra, n_dec = len(ra_centres), len(dec_centres)
print(f"RA  centres  : {n_ra}   [{ra_centres[0]:.1f}° … {ra_centres[-1]:.1f}°]")
print(f"Dec centres  : {n_dec}   [{dec_centres[0]:.1f}° … {dec_centres[-1]:.1f}°]")
print(f"Total patches : {n_ra * n_dec}")
print()
print("RA_IDX  →  RA centre")
for i, r in enumerate(ra_centres):
    print(f"  {i}  →  {r:+.1f}°")
print()
print("DEC_IDX  →  Dec centre")
for j, d in enumerate(dec_centres):
    print(f"  {j}  →  {d:+.1f}°")

#### Unmasked patches

Inspect the raw map first — spot bright point sources, decide which mask to use below.

In [ ]:
# Choose patch
#   RA_IDX  : 0 … n_ra-1   (west → east)
#   DEC_IDX : 0 … n_dec-1  (south → north)
RA_IDX  = 0
DEC_IDX = 0

ra_c  = ra_centres[RA_IDX]
dec_c = dec_centres[DEC_IDX]
print(f"Patch ({RA_IDX}, {DEC_IDX})  →  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°")

In [ ]:
# Plot (unmasked)
rms        = float(np.std(stokes_arr[obs_mask]))
vmin, vmax = -rms, rms

show_map_thumbnail(
    stokes_arr,
    vmin=vmin, vmax=vmax,
    title=(
        f"{STOKES_KEY}  unmasked  |  patch ({RA_IDX}, {DEC_IDX})  "
        f"|  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°"
    ),
    unit="Tcmb", cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")

#### Load mask

After inspecting the raw patches above, choose a mask from `MASK_FILES` and run this cell.

In [ ]:
# Choose mask
# Available keys (defined in §1):
#   mask_250_30   — 250 mJy point-source mask, 30 arcmin apodisation
#   mask_250_60   — 250 mJy point-source mask, 60 arcmin apodisation
#   mask_250_nd30 — 250 mJy, no disk, 30 arcmin
#   mask_250_nd60 — 250 mJy, no disk, 60 arcmin
#   mask_100_30   — 100 mJy point-source mask, 30 arcmin
#   mask_apod_30  — apodisation only, 30 arcmin (no source masking)
#   mask_apod_60  — apodisation only, 60 arcmin (no source masking)
ACTIVE_MASK = "mask_250_nd30"

mask_path = os.path.join(MASK_DIR, MASK_FILES[ACTIVE_MASK])
with np.load(mask_path) as d:
    apod     = d[d.files[0]].astype(float)
    mask_key = d.files[0]

nside_mask = hp.get_nside(apod)
if nside_mask != nside:
    print(f"Upgrading mask from nside={nside_mask} → nside={nside}")
    apod = hp.ud_grade(apod, nside_out=nside)

obs_and_mask = obs_mask & (apod > 0)

# Apply apodisation weight
stokes_masked                 = stokes_arr.copy()
stokes_masked[obs_and_mask]  *= apod[obs_and_mask]
stokes_masked[~obs_and_mask]  = hp.UNSEEN

print(f"Mask loaded  : {ACTIVE_MASK}")
print(f"File         : {MASK_FILES[ACTIVE_MASK]}")
print(f"key='{mask_key}'  |  nside={hp.get_nside(apod)}")
print(f"Pixels after mask : {obs_and_mask.sum():,}  (was {obs_mask.sum():,} unmasked)")

#### Masked patches

Same `RA_IDX` / `DEC_IDX` as above — easy before/after comparison.

In [ ]:
# Choose patch
#   Same RA_IDX / DEC_IDX as §5 for direct before/after comparison
RA_IDX  = 0
DEC_IDX = 0

ra_c  = ra_centres[RA_IDX]
dec_c = dec_centres[DEC_IDX]
print(f"Patch ({RA_IDX}, {DEC_IDX})  →  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°")

In [ ]:
# Plot (masked)
rms        = float(np.std(stokes_masked[obs_and_mask]))
vmin, vmax = -rms, rms

show_map_thumbnail(
    stokes_masked,
    vmin=vmin, vmax=vmax,
    title=(
        f"{STOKES_KEY}  masked ({ACTIVE_MASK})  |  patch ({RA_IDX}, {DEC_IDX})  "
        f"|  RA {ra_c:+.1f}°  Dec {dec_c:+.1f}°"
    ),
    unit="Tcmb", cmap=CMAP,
    rot=(ra_c, dec_c, 0),
    xsize=PATCH_PIX, ysize=PATCH_PIX,
    reso=RESO_ARCMIN,
)
plt.show()

In [ ]:
# Close
plt.close("all")

#### Power spectrum (draft)

> Draft only — no beam correction, no mode-coupling / purification.

In [ ]:
# Load Q and U for E/B decomposition
# This is independent of STOKES_KEY — E/B decomposition needs both simultaneously.
# Requires §6 (load mask) to have been run first so apod and obs_and_mask exist.

Q_arr = hp.read_map(COADD_FILE, field=1, partial=False)
U_arr = hp.read_map(COADD_FILE, field=2, partial=False)

# Apply the same apodisation mask
Q_masked = Q_arr.copy()
U_masked = U_arr.copy()
Q_masked[obs_and_mask] *= apod[obs_and_mask]
U_masked[obs_and_mask] *= apod[obs_and_mask]
Q_masked[~obs_and_mask] = 0.0   # anafast expects 0, not hp.UNSEEN
U_masked[~obs_and_mask] = 0.0

# hp.anafast needs [T, Q, U] to decompose into E/B.
# We only care about EE and BB, so pass T=zeros.
T_zeros = np.zeros(len(Q_arr))

LMAX = 1024
cls = hp.anafast([T_zeros, Q_masked, U_masked], lmax=LMAX)
# cls = [TT, EE, BB, TE, EB, TB]
cl_EE = cls[1]
cl_BB = cls[2]
ell   = np.arange(len(cl_BB))

print(f"Computed pseudo-C_ell up to ell={LMAX}")
print(f"Note: mask mixes E into B — BB is overestimated without mode-coupling correction.")

In [ ]:
# Plot BB pseudo-D_ell
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ell[2:], ell[2:] * (ell[2:] + 1) * cl_BB[2:] / (2 * np.pi), label="BB")
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell^{BB}$  [$\mu{\rm K}^2$]")
ax.set_title(
    f"BB pseudo-$C_\\ell$  |  {os.path.basename(COADD_FILE)}  |  mask: {ACTIVE_MASK}\n"
    f"(draft — no beam correction, no E→B leakage correction)"
)
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Save C_ell to FITS — uncomment when ready
# out_cl = os.path.join(SAVE_DIR, f"cl_{STOKES_KEY}_{ACTIVE_MASK}.fits")
# hp.write_cl(out_cl, cl, overwrite=True)
# print(f"C_ell saved → {out_cl}")